In [31]:
import pandas as pd
from pathlib import Path

# ============================== Config ==============================
ROOT = Path().resolve().parents[1]
MILLAGE_FP = ROOT / "data" / "Township_Millage_Table.csv"
OUTPUT_DIR = ROOT / "outputs"
FUNDS = ["FireRate", "RoadRate"]

YEARS_FOR_AVG_PER_FUND = {
    "FireRate": {2014, 2017, 2020, 2023},
    "RoadRate": {2017, 2020, 2023},
}

# ================== Township totals & avg %Δ helper =================
def build_township_from_taxable_dict(taxable_dict):
    rows = []
    for (muni, pclass), yearly in taxable_dict.items():
        for year, val in yearly.items():
            tv = float(val[0]) + float(val[1]) if isinstance(val, (tuple, list)) and len(val) == 2 else float(val)
            rows.append({"Municipality": muni, "PropertyClass": pclass, "TaxYear": int(year), "TaxableValue": tv})
    df_tx = pd.DataFrame(rows)

    township_totals = (
        df_tx.groupby(["TaxYear", "PropertyClass"], as_index=False)
             .agg(TaxableValue=("TaxableValue", "sum"))
             .sort_values(["PropertyClass", "TaxYear"])
    )

    township_totals["TaxableValue_pct_change"] = (
        township_totals.groupby("PropertyClass", sort=False)["TaxableValue"].pct_change() * 100.0
    )

    all_years = set().union(*YEARS_FOR_AVG_PER_FUND.values())
    mask_years = township_totals["TaxYear"].isin(all_years)
    avg_tx_change = (
        township_totals[mask_years]
        .dropna(subset=["TaxableValue_pct_change"])
        .groupby("PropertyClass", as_index=False)
        .agg(avg_tx_pct_change=("TaxableValue_pct_change", "mean"))
    )
    return township_totals, avg_tx_change

# ============================== Millage =============================
df = pd.read_csv(MILLAGE_FP)

required_cols = {"TaxYear", "PropertyClass"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Millage table missing required column(s): {missing}")

fund_cols_found = [c for c in FUNDS if c in df.columns]
if not fund_cols_found:
    raise ValueError(f"No requested fund columns found. Looked for: {FUNDS}")

pc_map = {"Res/Agr": "ResAgr", "Residential": "ResAgr", "Com/Ind": "ComInd", "Commercial": "ComInd"}
df["PropertyClass"] = df["PropertyClass"].replace(pc_map)

# =========================== Taxable values =========================
taxable_data = {
    ("Jefferson Unincorporated", "ResAgr"): {
        2014: (407_807_070, 2_731_540), 2015: (389_535_970, 2_893_880),
        2016: (394_781_780, 3_114_460), 2017: (448_618_560, 3_264_290),
        2018: (457_955_400, 3_368_530), 2019: (477_750_550, 5_763_030),
        2020: (568_049_370, 3_308_040), 2021: (585_038_010, 3_641_900),
        2022: (576_718_700, 3_603_720), 2023: (768_096_650, 4_056_820),
        2024: (763_057_600, 3_177_510),
    },
    ("Jefferson Unincorporated", "ComInd"): {
        2014: (18_498_090, 36_028_230), 2015: (8_989_220, 11_740_430),
        2016: (8_814_320, 8_106_700),  2017: (9_816_490, 9_176_010),
        2018: (15_470_630, 9_214_410), 2019: (11_671_440, 9_214_410),
        2020: (12_940_970, 10_286_530),2021: (13_639_040, 10_633_240),
        2022: (13_244_310, 10_148_800),2023: (26_236_970, 12_954_810),
        2024: (16_847_860, 12_925_410),
    },
    ("Gahanna", "ResAgr"): {
        2014: (2_705_600, 98_970), 2015: (2_766_850, 98_970), 2016: (2_755_050, 98_970),
        2017: (2_730_400, 99_530), 2018: (2_823_610, 99_530), 2019: (3_152_540, 99_530),
        2020: (4_121_040, 89_370), 2021: (4_182_260, 8_990),  2022: (4_165_320, 0),
        2023: (5_379_930, 0),      2024: (5_347_560, 0),
    },
    ("Gahanna", "ComInd"): {
        2014: (457_120, 1_650),  2015: (460_340, 1_650),  2016: (651_580, 1_650),
        2017: (626_240, 1_650),  2018: (1_980_250, 1_650),2019: (2_238_700, 1_650),
        2020: (2_511_240, 1_890),2021: (3_015_130, 1_890),2022: (3_514_270, 603_650),
        2023: (8_013_600, 830_240), 2024: (5_418_120, 546_950),
    },
    ("Reynoldsburg", "ResAgr"): {
        2014: (16_729_200, 0), 2015: (16_778_790, 0), 2016: (16_785_700, 0),
        2017: (19_159_550, 0), 2018: (19_096_040, 0), 2019: (19_155_550, 0),
        2020: (23_725_480, 0), 2021: (23_762_440, 0), 2022: (23_810_850, 0),
        2023: (33_284_200, 0), 2024: (33_195_740, 0),
    },
    ("Reynoldsburg", "ComInd"): {
        2014: (8_723_060, 24_367_000), 2015: (8_243_170, 24_367_000),
        2016: (8_243_170, 24_395_040), 2017: (10_733_250, 24_531_500),
        2018: (14_170_450, 24_531_500), 2019: (15_314_850, 24_531_500),
        2020: (17_743_950, 25_936_270), 2021: (13_750_060, 25_936_240),
        2022: (13_750_060, 25_936_240), 2023: (11_455_830, 24_531_550),
        2024: (11_455_830, 24_531_550),
    },
    ("Columbus", "ResAgr"): {
        2014: (4_214_470, 0), 2015: (4_573_250, 0), 2016: (5_264_870, 0),
        2017: (6_131_530, 0), 2018: (7_966_660, 0), 2019: (9_122_090, 0),
        2020: (12_079_200, 0), 2021: (12_103_750, 0), 2022: (12_132_270, 0),
        2023: (17_026_390, 0), 2024: (17_042_400, 0),
    },
}

township_totals, avg_tx_change = build_township_from_taxable_dict(taxable_data)

# ========================= Fund rate series =========================
assert {"TaxYear", "PropertyClass", "Municipality"}.issubset(df.columns), "Millage file missing required columns."

# Fire: mean across municipalities
fire_series = (
    df.loc[df["FireRate"].notna(), ["TaxYear", "PropertyClass", "FireRate"]]
      .groupby(["TaxYear", "PropertyClass"], as_index=False)
      .agg(Rate=("FireRate", "mean"))
)
fire_series["Fund"] = "FireRate"

# Road: Jefferson Unincorporated only
road_mask = (df["Municipality"] == "Jefferson Unincorporated") & df["RoadRate"].notna()
road_series = df.loc[road_mask, ["TaxYear", "PropertyClass", "RoadRate"]].rename(columns={"RoadRate": "Rate"})
road_series["Fund"] = "RoadRate"

rate_long = pd.concat([fire_series, road_series], ignore_index=True).sort_values(["Fund", "PropertyClass", "TaxYear"])

rate_long["Rate_pct_change"] = rate_long.groupby(["Fund", "PropertyClass"])["Rate"].pct_change() * 100.0

# Avg %Δ with fund-specific anchor years
avg_rate_change = pd.concat(
    [
        rate_long[(rate_long["Fund"] == "FireRate") & (rate_long["TaxYear"].isin(YEARS_FOR_AVG_PER_FUND["FireRate"]))],
        rate_long[(rate_long["Fund"] == "RoadRate") & (rate_long["TaxYear"].isin(YEARS_FOR_AVG_PER_FUND["RoadRate"]))],
    ],
    ignore_index=True,
)
avg_rate_change = (
    avg_rate_change.dropna(subset=["Rate_pct_change"])
                   .groupby(["Fund", "PropertyClass"], as_index=False)
                   .agg(avg_rate_pct_change=("Rate_pct_change", "mean"))
)

# ============================== Elasticity ==========================
elas = avg_rate_change.merge(avg_tx_change, on="PropertyClass", how="inner", validate="many_to_one")
elas["Elasticity"] = elas.apply(
    lambda r: (r["avg_rate_pct_change"] / r["avg_tx_pct_change"])
    if pd.notnull(r["avg_rate_pct_change"]) and pd.notnull(r["avg_tx_pct_change"]) and r["avg_tx_pct_change"] != 0
    else pd.NA,
    axis=1,
)

# ============================ TIF inputs ============================
tif_totals_input = {
    ("170", "ComInd"): {2018: 556_500, 2019: 1_930_250, 2020: 0, 2021: 5_773_710, 2022: 3_765_340, 2023: 4_237_520, 2024: 0},
    ("170", "ResAgr"): {2018: 3_573_990, 2019: 7_414_530, 2020: 16_842_520, 2021: 26_371_200, 2022: 47_234_790, 2023: 77_962_220, 2024: 88_443_750},
    ("171", "ComInd"): {2018: 3_191_620, 2019: 4_621_960, 2020: 5_546_350, 2021: 5_546_350, 2022: 5_546_350, 2023: 6_282_850, 2024: 6_282_850},
    ("175", "ResAgr"): {2014: 8_338_510, 2015: 8_314_490, 2016: 8_305_140, 2017: 10_176_090, 2018: 10_176_610, 2019: 10_188_200, 2020: 12_204_230,
                        2021: 12_240_320, 2022: 12_242_800, 2023: 16_311_530, 2024: 16_326_760},
    ("027", "ResAgr"): {2014: 4_629_980, 2015: 5_714_610, 2016: 6_822_970, 2017: 8_426_810, 2018: 8_570_380, 2019: 8_577_790, 2020: 10_011_690,
                        2021: 10_012_490, 2022: 10_037_490, 2023: 10_477_460, 2024: 10_520_890},
    ("067", "ComInd"): {2019: 181_930, 2020: 1_679_500, 2021: 2_245_830, 2022: 2_245_830, 2023: 8_755_270, 2024: 8_447_730},
}
taxcode_to_muni = {"170": "Jefferson Unincorporated", "171": "Jefferson Unincorporated", "027": "Gahanna", "067": "Reynoldsburg", "175": "Columbus"}

rows = []
for (taxcode, pclass), yearly in tif_totals_input.items():
    for year, val in yearly.items():
        rows.append({
            "TaxYear": int(year),
            "TaxCode": str(taxcode),
            "Municipality": taxcode_to_muni.get(str(taxcode), "Other"),
            "PropertyClass": pclass,
            "TIF_AssessedImpr": float(val),
        })
tif_manual = pd.DataFrame(rows)
tif_manual = tif_manual[tif_manual["Municipality"] != "Other"].copy()

# =============== TIF share of base (class × year) ===================
township_totals2 = township_totals.rename(columns={"TaxableValue": "Total_Township_TaxableValue"}).copy()
tif_township_by_class = (
    tif_manual.groupby(["TaxYear", "PropertyClass"], as_index=False)
              .agg(TIF_Township_AssessedImpr=("TIF_AssessedImpr", "sum"))
)
pc_year_share = township_totals2.merge(tif_township_by_class, on=["TaxYear", "PropertyClass"], how="left")
pc_year_share["TIF_Township_AssessedImpr"] = pc_year_share["TIF_Township_AssessedImpr"].fillna(0.0)
pc_year_share["TIF_Share_of_Base"] = pc_year_share["TIF_Township_AssessedImpr"] / pc_year_share["Total_Township_TaxableValue"]

# ================== Factors & adjusted fund rates ==================
rate_pivot = rate_long.pivot_table(index=["TaxYear", "PropertyClass"], columns="Fund", values="Rate").reset_index()

out_rows = []
for _, row in pc_year_share.iterrows():
    tax_year = row["TaxYear"]
    pclass = row["PropertyClass"]
    tif_share = float(row["TIF_Share_of_Base"]) if pd.notna(row["TIF_Share_of_Base"]) else 0.0

    base_rates = rate_pivot[(rate_pivot["TaxYear"] == tax_year) & (rate_pivot["PropertyClass"] == pclass)]
    if base_rates.empty:
        continue
    base_rates = base_rates.iloc[0]

    for fund in fund_cols_found:
        e_lookup = elas[(elas["Fund"] == fund) & (elas["PropertyClass"] == pclass)]
        e_val = e_lookup["Elasticity"].iloc[0] if not e_lookup.empty else pd.NA
        rate_val = base_rates.get(fund, pd.NA)

        factor = (1.0 + float(e_val) * tif_share) if pd.notna(e_val) else pd.NA
        adj_rate = (float(rate_val) * factor) if (pd.notna(rate_val) and pd.notna(factor)) else pd.NA

        out_rows.append({
            "TaxYear": tax_year,
            "PropertyClass": pclass,
            "Fund": fund,
            "Elasticity": e_val,
            "TIF_Share_of_Base": tif_share,
            "Factor": factor,
            "Rate": rate_val,
            "AdjustedRate": adj_rate,
        })

factors = pd.DataFrame(out_rows).sort_values(["Fund", "PropertyClass", "TaxYear"])

# ==================== Losses (muni × year × fund) ==================
losses = tif_manual.merge(factors[["TaxYear", "PropertyClass", "Fund", "AdjustedRate"]], on=["TaxYear", "PropertyClass"], how="left")

# Road applies only to Jefferson Unincorporated
losses["AdjustedRate"] = losses.apply(
    lambda r: r["AdjustedRate"] if not (r["Fund"] == "RoadRate" and r["Municipality"] != "Jefferson Unincorporated") else 0.0,
    axis=1,
)

losses["Estimated_Loss"] = pd.to_numeric(losses["AdjustedRate"], errors="coerce") * (pd.to_numeric(losses["TIF_AssessedImpr"], errors="coerce") / 1000.0)

muni_year_fund_loss = (
    losses.groupby(["Municipality", "TaxYear", "Fund"], as_index=False)
          .agg(Estimated_Loss=("Estimated_Loss", "sum"),
               TIF_AssessedImpr=("TIF_AssessedImpr", "sum"))
          .sort_values(["Municipality", "TaxYear", "Fund"])
)

# ============================ Save output ===========================
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
losses_by_year_muni = (
    losses.groupby(["TaxYear", "Municipality", "Fund"], as_index=False)
          .agg(EstimatedLoss=("Estimated_Loss", "sum"))
          .sort_values(["TaxYear", "Municipality", "Fund"])
)
output_path = OUTPUT_DIR / "Estimated_Losses_ByYear_Municipality_Fire_Road.xlsx"
losses_by_year_muni.to_excel(output_path, index=False)
print(f"Saved: {output_path}")


[Diagnostics] Taxable %Δ values used in averages by class:
PropertyClass      contrib_years
       ComInd [2017, 2020, 2023]
       ResAgr [2017, 2020, 2023]

=== Average % change in TAXABLE VALUE (using union of target years) ===
PropertyClass  avg_tx_pct_change
       ComInd          14.861623
       ResAgr          21.887724

=== Average % change in FUND RATES (Fire/Road; fund-specific years) ===
    Fund PropertyClass  avg_rate_pct_change
FireRate        ComInd            -8.636800
FireRate        ResAgr           -16.296911
RoadRate        ComInd           -13.040916
RoadRate        ResAgr           -14.522780

=== Elasticity by Fund × PropertyClass (Fire uses 2014/17/20/23; Road uses 2017/20/23) ===
    Fund PropertyClass  avg_rate_pct_change  avg_tx_pct_change  Elasticity
FireRate        ComInd            -8.636800          14.861623   -0.581148
FireRate        ResAgr           -16.296911          21.887724   -0.744569
RoadRate        ComInd           -13.040916          14.861